# Strings and trustworthy text data

Learn how Python represents, inspects, transforms, parses, validates, and formats text while
preserving the original evidence.

**Lecture 1 · Python Foundations I · CMOR 438 / INDE 577**


## How to use this notebook

**Estimated time:** 40 minutes of core instruction, plus 25–35 minutes of practice and extension.

**Prerequisite:** notebook 01's distinction among values, types, names, and objects.

Follow the **Core** cells during class. Pause at every prediction prompt before executing it.
**Practice** cells include executable success criteria. **Extension** cells deepen Unicode,
encoding, and policy questions and may be completed after class.

This is not a catalogue of string methods. It develops a model for deciding which text
transformation is justified, observing its effect, and checking that it did not corrupt meaning.


## Learning objectives

By the end of this notebook, you should be able to:

- explain why a Python string is an immutable sequence of Unicode code points;
- distinguish source text, its `repr`, encoded bytes, and displayed output;
- predict indexing and half-open slicing operations;
- select among `strip`, prefix/suffix removal, `split`, `partition`, `join`, and `replace`;
- parse a small scientific log record without discarding the raw record;
- normalize case and Unicode only under an explicit data contract;
- format numeric results for people without changing the calculation; and
- diagnose text defects using `type`, `repr`, `len`, assertions, and code-point inspection.


## Why this matters in industry

Text is often the first representation of data: filenames, experiment identifiers, column names,
instrument logs, categories, command-line arguments, API payloads (data sent through a documented
software interface), and human annotations. A string
can *look* like a number, Boolean, date, or missing value while still being only text.

Most dangerous text bugs are valid Python. An aggressive replacement can merge two identifiers;
an invisible character can break a join; a wrong encoding can make a file unreadable; and a model
score formatted for a report can accidentally become the value used in later computation.

Reliable text work separates three stages:

```text
raw evidence → contract-aware parsing and normalization → typed internal value
```

Each arrow is a decision that should be explainable and testable.


## Scientific question and running scenario

A spectroscopy screening system emits one compact event line:

```text
 EXP-0042 | wavelength_nm=532.0 | score=0.873 | decision=REVIEW 
```

We want to answer: **Which experiment and wavelength produced the model score, and what action was
requested?** The fields represent different mathematical objects even though the transport format
is text:

| Field | Meaning | Desired Python representation |
| --- | --- | --- |
| experiment ID | categorical identifier | normalized `str` |
| wavelength | physical quantity in nanometers | `float` plus an explicit unit |
| model score | bounded numerical output | validated `float` |
| decision | controlled category | normalized `str` |

The example calls `0.873` a **score**, not a probability. A score in `[0, 1]` is not necessarily
calibrated and requires a separately justified decision threshold.


## Professional practice: raw text and canonical text serve different jobs

| Data scientist asks | Software engineer asks |
| --- | --- |
| What phenomenon does each field represent? | What grammar and type does each field promise? |
| Are units and missing-value conventions known? | Where are conversion failures reported? |
| Could normalization merge distinct samples? | Are raw and canonical values both traceable? |
| Does the score have a calibrated interpretation? | Are bounds and accepted categories tested? |

Preserve `raw_event` as evidence. Derive new names for parsed values. Do not repeatedly mutate one
ambiguous variable called `data`, and do not treat cleanup as harmless merely because it is short.


In [ ]:
raw_event = " EXP-0042 | wavelength_nm=532.0 | score=0.873 | decision=REVIEW "

print(raw_event)
print(repr(raw_event))

assert isinstance(raw_event, str)
assert raw_event.startswith(" ")
assert raw_event.endswith(" ")


## Core: a string is an immutable Unicode sequence

Python's `str` represents Unicode text. Conceptually, it is an ordered sequence of code points.
That sequence model explains why strings support `len`, indexing, slicing, membership tests, and
iteration.

**Immutable** means the existing string value cannot change in place. Operations such as
`.strip()` and `.replace()` create new strings. Immutability makes strings safe dictionary keys and
prevents an alias from observing a hidden in-place edit.

Printed characters are not the same as the underlying representation. Use `repr(text)` when spaces,
newlines, tabs, quotes, or escape sequences may matter.


In [ ]:
sample_label = "  line\tA\n"

print("display with print:")
print(sample_label)
print("representation with repr:", repr(sample_label))

assert len(sample_label) == 9
assert sample_label[0] == " "
assert sample_label[-1] == "\n"


## Core: literals describe string values in source code

Single and double quotes create the same type; choose the delimiter that makes the content clear.
Triple-quoted strings may span lines. A backslash begins an escape such as `
` (newline), `	`
(tab), or `\` (one literal backslash).

A raw string literal such as `r"C:\data\run-01"` changes how Python source interprets
backslashes. It does **not** clean external input, and a raw string cannot end with one unpaired
backslash. For filesystem paths, prefer `pathlib.Path` over manual separator manipulation.


In [ ]:
single_quoted = 'instrument "A"'
double_quoted = "instrument 'A'"
multiline_note = """first observation
second observation"""
windows_example = r"C:\data\run-01"

assert type(single_quoted) is str
assert multiline_note.count("\n") == 1
assert windows_example.endswith("run-01")

print(single_quoted)
print(double_quoted)
print(repr(multiline_note))
print(windows_example)


## Core: indexing selects; slicing extracts

Positions start at zero. Negative positions count backward from the end. A slice
`text[start:stop:step]` uses a **half-open** interval: it includes `start` and excludes `stop`.
Omitted bounds extend to the corresponding end.

```text
text:   E X P - 0 0 4 2
index:  0 1 2 3 4 5 6 7
        -8             -1
```

Indexing outside the sequence raises `IndexError`. Slicing beyond an endpoint safely stops at the
available boundary. Neither operation changes the source string.


In [ ]:
experiment_id = "EXP-0042"

assert experiment_id[0] == "E"
assert experiment_id[-1] == "2"
assert experiment_id[:3] == "EXP"
assert experiment_id[4:] == "0042"
assert experiment_id[::2] == "EP04"
assert experiment_id[::-1] == "2400-PXE"
assert experiment_id[:100] == experiment_id


### Practice: predict the sequence operations

Before executing the next cell, write the value and type produced by each expression:

1. `channel[0]`
2. `channel[-2:]`
3. `channel[1:4]`
4. `"nm" in channel`
5. `channel[99]`

The final operation is captured so the notebook remains executable.


In [ ]:
channel = "532nm"

print(channel[0])
print(channel[-2:])
print(channel[1:4])
print("nm" in channel)

try:
    channel[99]
except IndexError as error:
    print(f"Captured {type(error).__name__}: {error}")

assert channel[-2:] == "nm"
assert channel[1:4] == "32n"
assert "nm" in channel


## Core: string methods return new values

An indexed position cannot be assigned because strings are immutable. Bind the result of a method
or expression to a new, meaningfully named variable. Keeping raw and derived names side by side
makes provenance visible.

The captured error below is intentional. Catching it lets us study the failure without breaking a
top-to-bottom notebook run.


In [ ]:
raw_identifier = " exp-0042 "

try:
    raw_identifier[0] = "E"
except TypeError as error:
    print(f"Captured {type(error).__name__}: {error}")

clean_identifier = raw_identifier.strip().upper()

assert raw_identifier == " exp-0042 "
assert clean_identifier == "EXP-0042"


## Core: trim boundaries without pretending to repair everything

`text.strip()` removes whitespace from both ends; `lstrip()` and `rstrip()` affect one end. It does
not remove internal whitespace. That narrow behavior is useful at an ingestion boundary.

An argument changes the meaning: `text.strip(".csv")` removes *any run of those characters* from
both ends. It does not remove one exact suffix. Use `removeprefix` or `removesuffix` when an exact
affix is the contract.


In [ ]:
raw_filename = "  class.csv  "
trimmed_filename = raw_filename.strip()
stem = trimmed_filename.removesuffix(".csv")
strip_trap = trimmed_filename.strip(".csv")

assert trimmed_filename == "class.csv"
assert stem == "class"
assert strip_trap == "la"

print("exact suffix removal:", stem)
print("character-set stripping:", strip_trap)


## Core: case normalization requires a contract

Use `.lower()` for ordinary display-oriented lowercasing. Use `.casefold()` when the contract calls
for caseless comparison across writing systems; it is intentionally more aggressive. Neither is a
universal identifier policy.

Case normalization is appropriate only if upstream declares the field case-insensitive. Never
apply `.title()` to people's names as a supposed correction: names and writing systems do not obey
one capitalization rule.


In [ ]:
incoming_category = "  ReViEw "
category_key = incoming_category.strip().casefold()

german_word = "Straße"

assert category_key == "review"
assert german_word.lower() == "straße"
assert german_word.casefold() == "strasse"
assert incoming_category == "  ReViEw "


## Core: split only on delimiters guaranteed by the format

`split(delimiter)` returns pieces and may produce more pieces than expected. `maxsplit` limits how
many divisions occur. With no argument, `split()` treats runs of whitespace as separators.

`partition(delimiter)` always returns a three-item tuple:
`(before, delimiter, after)`. The empty delimiter field reveals that the expected separator was not
found. This is often clearer than splitting when a field has one key/value boundary.

Delimiter-based parsing is appropriate only for a format whose grammar guarantees that delimiter
cannot appear unescaped inside a value. Use `csv`, `json`, or another real parser for real formats.


In [ ]:
raw_field = " wavelength_nm=532.0 "
clean_field = raw_field.strip()
field_name, separator, field_text = clean_field.partition("=")

assert separator == "="
assert field_name == "wavelength_nm"
assert field_text == "532.0"

note_words = "signal   requires  review".split()
assert note_words == ["signal", "requires", "review"]

print(field_name, repr(field_text))


## Core: `join` makes the separator own the operation

`separator.join(pieces)` combines an iterable of strings. This design reads as “join these pieces
using this separator.” All pieces must already be strings; requiring explicit conversion avoids
guessing how scientific values should be represented.

Repeated `+` is fine for a few fixed fragments. `join` is clearer for a collection and avoids
building many intermediate strings in a large loop.


In [ ]:
label_parts = ["EXP-0042", "532.0nm", "review"]
compact_label = " | ".join(label_parts)

assert compact_label == "EXP-0042 | 532.0nm | review"
print(compact_label)

try:
    " | ".join(["EXP-0042", 532.0])
except TypeError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: replacement is literal and potentially overbroad

`replace(old, new)` replaces every non-overlapping match unless a count is supplied. It knows
nothing about tokens, words, units, or identifiers. First ask whether the location and number of
matches are part of the data contract.

Prefer an exact prefix/suffix operation or structured parser when only one structural component may
change. Preserve the original before any lossy transformation.


In [ ]:
raw_note = "sensor A; backup sensor A unavailable"
all_replaced = raw_note.replace("sensor A", "sensor B")
first_replaced = raw_note.replace("sensor A", "sensor B", 1)

assert all_replaced.count("sensor B") == 2
assert first_replaced == "sensor B; backup sensor A unavailable"
assert raw_note == "sensor A; backup sensor A unavailable"


## Core: text that resembles a number is still text

String predicates such as `.isdecimal()` answer narrow character questions. They are not complete
scientific-number validators: signs, decimal points, exponents, missing-value markers, and units all
need a grammar.

At a declared boundary, parse with `int` or `float`, catch conversion errors, then validate the
domain separately. Successful parsing proves only that syntax was accepted—not that the value has
the right unit, range, or scientific meaning.


In [ ]:
score_text = "0.873"
score = float(score_text)

assert isinstance(score_text, str)
assert isinstance(score, float)
assert 0.0 <= score <= 1.0

invalid_score_text = "not-recorded"
try:
    float(invalid_score_text)
except ValueError as error:
    print(f"Rejected {invalid_score_text!r}: {error}")


## Worked example: parse one model-screening event

Now apply the mental model to `raw_event`:

1. preserve the original line;
2. divide only at the documented field delimiter;
3. trim boundary whitespace from each field;
4. verify each key/value separator and expected key;
5. convert numerical text and validate its domain; and
6. create presentation text from typed values.

This hand-written parser is intentionally small and transparent. It is not a replacement for CSV,
JSON, or a production schema library. Lecture 2 loads real interchange formats with standard-library
parsers.


In [ ]:
fields = raw_event.split("|")
assert len(fields) == 4

experiment_text = fields[0].strip()
wavelength_key, wavelength_separator, wavelength_text = fields[1].strip().partition("=")
score_key, score_separator, event_score_text = fields[2].strip().partition("=")
decision_key, decision_separator, decision_text = fields[3].strip().partition("=")

assert wavelength_separator == score_separator == decision_separator == "="
assert (wavelength_key, score_key, decision_key) == (
    "wavelength_nm",
    "score",
    "decision",
)

experiment_key = experiment_text.casefold()
wavelength_nm = float(wavelength_text)
event_score = float(event_score_text)
decision = decision_text.casefold()

assert experiment_key == "exp-0042"
assert wavelength_nm == 532.0
assert 0.0 <= event_score <= 1.0
assert decision in ("review", "accept", "reject")
assert raw_event.startswith(" ")


## Core: formatting is a presentation boundary

F-strings interpolate values into text. Format specifications control decimal places, percentages,
alignment, and separators. Formatting creates a new string; it must not replace the numerical value
used for calculation.

Choose precision to communicate the instrument or analysis, not merely because more digits fit on
screen. Here `.1f` is a hypothetical reporting rule, while `.3f` preserves the score's supplied
precision.


In [ ]:
summary = (
    f"experiment={experiment_key} | wavelength={wavelength_nm:.1f} nm | "
    f"score={event_score:.3f} | decision={decision}"
)

print(summary)
print(f"diagnostic: {wavelength_text=} {event_score_text=}")

assert summary == (
    "experiment=exp-0042 | wavelength=532.0 nm | "
    "score=0.873 | decision=review"
)
assert event_score == 0.873


## Extension: Unicode equality and human-visible characters

Two strings can display identically while using different code-point sequences. Unicode
normalization can establish a consistent representation when the contract requires canonical
equivalence. NFC is a common conservative choice; compatibility normalization can erase meaningful
distinctions and requires stronger justification.

Also, `len(text)` counts Python's Unicode code points, not necessarily user-perceived characters.
Emoji, combining marks, and writing systems can form one visible grapheme from multiple code points.
Do not use `len` as a universal display-width or human-name limit.


In [ ]:
import unicodedata

composed = "Café"
decomposed = "Café"
family_emoji = "👨‍👩‍👧‍👦"

assert composed != decomposed
assert unicodedata.normalize("NFC", composed) == unicodedata.normalize(
    "NFC", decomposed
)
assert len(family_emoji) > 1

print(repr(composed), [f"U+{ord(character):04X}" for character in composed])
print(repr(decomposed), [f"U+{ord(character):04X}" for character in decomposed])
print("family emoji code points:", len(family_emoji))


## Extension: text and bytes meet at an encoding boundary

Files and networks store bytes. Decoding interprets bytes using an encoding and produces `str`;
encoding converts `str` back to bytes. UTF-8 is the usual interoperable choice, but Python cannot
infer an undocumented encoding with certainty.

```text
bytes --decode with UTF-8--> str --encode with UTF-8--> bytes
```

Always state the encoding when reading or writing text files. A decoding failure is useful evidence
that the bytes and assumed contract disagree; replacing invalid bytes silently can destroy data.


In [ ]:
scientific_text = "λ = 532 nm"
encoded = scientific_text.encode("utf-8")
decoded = encoded.decode("utf-8")

assert isinstance(encoded, bytes)
assert decoded == scientific_text

print(repr(scientific_text))
print(encoded)

try:
    encoded.decode("ascii")
except UnicodeDecodeError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Debugging playbook for suspicious text

Use evidence in this order:

1. `type(value)` — is this actually `str`, `bytes`, or another object?
2. `repr(value)` — are whitespace, escapes, or quotes hidden by display?
3. `len(value)` and a small slice — where does the unexpected region begin?
4. `[f"U+{ord(c):04X}" for c in value]` — which code points are present?
5. delimiter and field counts — did parsing assumptions hold?
6. explicit assertions — which contract was violated?
7. raw evidence — can the transformation be audited or replayed?

Do not begin with a long chain of `.strip().lower().replace(...)`. Diagnose first, then adopt the
smallest transformation supported by the field's contract.


## Practice: guided filename parsing

An instrument writes `"  run-017__sensor-A__532.0nm.txt  "`. Derive:

- `run_id == "run-017"`;
- `sensor_id == "sensor-A"`; and
- `practice_wavelength_nm == 532.0`.

First trim whitespace, then remove the exact `.txt` suffix, split on the documented `__` delimiter,
remove the exact `nm` suffix, and convert the numerical text. The assertions are the success
criteria.


In [ ]:
practice_raw_filename = "  run-017__sensor-A__532.0nm.txt  "
practice_filename = practice_raw_filename.strip().removesuffix(".txt")
practice_parts = practice_filename.split("__")

assert len(practice_parts) == 3

run_id = practice_parts[0]
sensor_id = practice_parts[1]
practice_wavelength_text = practice_parts[2].removesuffix("nm")
practice_wavelength_nm = float(practice_wavelength_text)

assert run_id == "run-017"
assert sensor_id == "sensor-A"
assert practice_wavelength_nm == 532.0
assert practice_raw_filename.startswith("  ")


## Practice: independent controlled-category cleanup

Given `practice_raw_status = "\tNeeds Review \n"`, produce a canonical status under this contract:

- surrounding whitespace is insignificant;
- comparison is caseless;
- internal whitespace becomes one hyphen; and
- the only accepted results are `ready`, `needs-review`, and `rejected`.

Preserve the raw value. Your result must satisfy the provided assertions. Then explain why this same
policy should not automatically be applied to free-form scientist notes.


In [ ]:
practice_raw_status = "\tNeeds Review \n"
practice_status = "-".join(practice_raw_status.strip().casefold().split())
accepted_statuses = ("ready", "needs-review", "rejected")

assert practice_status == "needs-review"
assert practice_status in accepted_statuses
assert practice_raw_status == "\tNeeds Review \n"


## Extension: make and defend a normalization policy

Two laboratories use sample IDs `"Müller-07"`, `"Muller-07"`, and `"MUELLER-07"`. A teammate
proposes deleting non-ASCII characters and lowercasing everything before joining datasets.

Write a short design response addressing:

1. which identifiers could collapse together;
2. whether these strings are display labels or authoritative keys;
3. which upstream metadata or mapping table you need;
4. how you would measure affected records before changing policy;
5. which raw and canonical fields you would retain; and
6. what assertions or tests would detect accidental collisions.

There is no universal method chain that answers these questions. A strong response makes the
domain contract explicit and proposes an auditable migration.


## Common failure modes

| Symptom | Likely cause | Better response |
| --- | --- | --- |
| records that look equal do not join | whitespace, case, or Unicode differs | compare `repr`, code points, and source contracts |
| `bool("False")` is `True` | nonempty strings are truthy | enumerate and parse accepted categories |
| `strip(".csv")` damages a filename | argument treated as a set of characters | use `removesuffix(".csv")` |
| `split` creates too many fields | delimiter appears inside a value | use the format's parser or a documented `maxsplit` |
| every matching phrase changes | `replace` is global by default | validate count or use structural parsing |
| `join` raises `TypeError` | at least one item is not text | format each value explicitly |
| decoding fails | bytes do not match the assumed encoding | confirm source encoding; do not silently discard bytes |
| displayed character count seems wrong | code points differ from grapheme clusters | use Unicode-aware UI tooling when that count matters |

Language behavior is only half the diagnosis. The other half is whether the chosen transformation
matches the scientific and operational meaning of the field.


## Retrieval practice

Answer without executing code:

1. What does `repr` reveal that ordinary `print` may hide?
2. Why does a slice exclude its stop index?
3. What does string immutability imply about `.strip()`?
4. When is `partition("=")` clearer than `split("=")`?
5. Why is `strip(".csv")` not suffix removal?
6. What is the difference between a numeric string and a number?
7. Why can case or Unicode normalization merge distinct identifiers?
8. What is the relationship among bytes, an encoding, and `str`?
9. Why should a formatted score remain separate from the numerical score?
10. Which evidence should be preserved when parsing fails?


## Takeaway and next step

Strings are immutable Unicode sequences, but trustworthy text processing is not merely a sequence
of method calls. Preserve raw evidence, diagnose representation with `repr`, parse only against a
known grammar, normalize only under a documented policy, validate the resulting domain value, and
keep formatting separate from computation.

Notebook 03 introduces lists, tuples, and general sequence operations. The list returned by
`split` will become one example of a broader collection model.


## Further reading

- [Python text sequence type — `str`](https://docs.python.org/3.12/library/stdtypes.html#text-sequence-type-str)
- [Python string methods](https://docs.python.org/3.12/library/stdtypes.html#string-methods)
- [Python Unicode HOWTO](https://docs.python.org/3.12/howto/unicode.html)
- [Python lexical analysis: string literals](https://docs.python.org/3.12/reference/lexical_analysis.html#string-and-bytes-literals)
- [Python format specification mini-language](https://docs.python.org/3.12/library/string.html#formatspec)
- [Unicode normalization forms](https://unicode.org/reports/tr15/)

The standard-library documentation is a reference, not a recommended order for memorizing methods.
Return to it when a contract requires behavior not developed in this notebook.
